# Right Kidney Width

Original research notebook, cleaned for public review. Requires an authorized local dataset; no clinical records or historical outputs are supplied. Training and evaluation settings are retained as archived, not asserted to reproduce the manuscript.

In [ ]:
import numpy as np
import pandas as pd 
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint ,ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers.schedules import ExponentialDecay

def r2_metric(y_true, y_pred):
    ss_res = tf.reduce_sum(tf.square(y_true - y_pred))
    ss_tot = tf.reduce_sum(tf.square(y_true - tf.reduce_mean(y_true)))
    return 1 - ss_res / (ss_tot + tf.keras.backend.epsilon())

## Local data input

In [ ]:
from pathlib import Path
import os

# Supply an authorized CSV locally; do not commit it.
data_path = Path(os.environ.get("RENAL_DATA_PATH", "../data/cbc.csv"))
if not data_path.is_file():
    raise FileNotFoundError("Set RENAL_DATA_PATH to an authorized local dataset.")
df = pd.read_csv(data_path)

## Features and target

In [ ]:
X = df [['Age (years old)','Weight (kg)','Sex','HCT', 'HB', 'RBC', 'MCV', 'MCH', 'MCHC', 'RDW', 'WBC', 'PLT',
       'Neutrophils', 'Lymphocytes', 'Monocytes', 'Eosinophils']]
Y = df['RK.W']

## Train/test split and scaling

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.1, random_state=0)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Neural network training

In [ ]:
model = tf.keras.Sequential ([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],),kernel_regularizer=l2(0.02)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(512, activation='selu',kernel_regularizer=l2(0.008) ),
    
    tf.keras.layers.Dense(1024, activation='selu',kernel_regularizer=l2(0.005)  ),
    
    tf.keras.layers.Dense(256
                          , activation='gelu' ),
    
    tf.keras.layers.Dense(128, activation='selu',kernel_regularizer=l2(0.04) ),
    
    tf.keras.layers.Dense(256, activation='relu'),

    tf.keras.layers.Dense(1)
])
# callbacks
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',  # Monitor the validation loss
    factor=0.5,  # Reduce the learning rate by a factor of 0.5
    patience=5,  # Wait for 5 epochs without improvement before reducing the learning rate
    verbose=1,  # Print a message when the learning rate is reduced
    min_lr=1e-6  # The learning rate will not go below this value
)

checkpoint = ModelCheckpoint(
    'best_model_RKW.keras',  # File name to save the best model
    monitor='val_loss',  # Metric to monitor (can be 'val_loss' or 'val_accuracy', etc.)
    save_best_only=True,  # Only save the best model (based on the metric)
    verbose=1,  # Prints a message when saving the model
    save_weights_only=False,  # Set to False to save the entire model (architecture + weights)
    mode='min',  # 'min' for loss (we want to minimize the loss)
)

model.compile(optimizer='adam', loss='mean_squared_error')
history = model.fit(X_train_scaled, Y_train, epochs=100, batch_size=4, validation_split=0.1, callbacks = [checkpoint,lr_scheduler])

## Evaluation

In [ ]:
predictions = model.predict(X_test_scaled)
print (f'r2(test) :{r2_score (Y_test,predictions)} ')
print (f'r2(train) :{r2_score (Y_train,model.predict(X_train_scaled))} ')

## Reload locally trained checkpoint

In [ ]:
best_model = tf.keras.models.load_model('best_model_RKW.keras', compile=False)

predictions_best = best_model.predict(X_test_scaled)
print (f'r2_best(test) :{r2_score (Y_test,predictions_best)} ')
print (f'r2_best(train) :{r2_score (Y_train,best_model.predict(X_train_scaled))} ')

In [ ]:
# 1. Plot training and validation loss over epochs
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()



# 4. Calculate and print R² and MAE for the test set
r2 = r2_score(Y_test, predictions)
mae = mean_absolute_error(Y_test, predictions)

print(f'R² Score for TensorFlow Model: {r2}')
print(f'Mean Absolute Error (MAE) for TensorFlow Model: {mae}')

## Gradient boosting

In [ ]:
import xgboost as xgb
from sklearn.metrics import r2_score, mean_absolute_error

# Train XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=6)
xgb_model.fit(X_train_scaled, Y_train)

xgb_preds_train = xgb_model.predict(X_train_scaled)
xgb_preds_test = xgb_model.predict(X_test_scaled)

xgb_r2 = r2_score(Y_test, xgb_preds_test)
xgb_r2_train = r2_score(Y_train , xgb_preds_train)

print(f"XGBoost R² Score: {xgb_r2}")
print(f"XGBoost R² Train: {xgb_r2_train}")

## Ensemble inputs

In [ ]:
nn_preds_train = best_model.predict(X_train_scaled)
nn_preds_test = best_model.predict(X_test_scaled)
r2_score(Y_test, nn_preds_test)

## Stacking

In [ ]:
# Stack the predictions of NN and XGBoost
stacked_train_preds = np.column_stack((nn_preds_train, xgb_preds_train))
stacked_test_preds = np.column_stack((nn_preds_test, xgb_preds_test))

from sklearn.tree import DecisionTreeRegressor
meta_model = DecisionTreeRegressor(random_state=42)
meta_model.fit(stacked_train_preds, Y_train)

# Get final predictions from the meta-model
final_preds = meta_model.predict(stacked_test_preds)

# Evaluate the stacked model
r2 = r2_score(Y_test, final_preds)
print(f"R² score of stacked model: {r2}")

## Blending

In [ ]:
nn_preds_test = nn_preds_test.reshape(-1)

blended_preds = (nn_preds_test + xgb_preds_test) / 2

# Evaluate blended model
r2 = r2_score(Y_test, blended_preds)
print(f"R² score of blended model: {r2}")